In [ ]:
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz

# ── Load complete 2026 squads (built from official BCCI data) ──
squads = pd.read_csv('../data/squads_2026_complete.csv')
meta = pd.read_csv('../data/player_metadata.csv')
scouting = pd.read_csv('../data/scouting_profiles.csv')

print(f"Complete squads: {len(squads)} players across {squads['team'].nunique()} teams")

# ── Rename team column to match rest of pipeline ──
squads.rename(columns={'team': 'current_franchise', 'role': 'clean_role'}, inplace=True)

# ── Merge metadata (batting_hand, bowling_style, batting_role, etc.) ──
# Fuzzy match squad names to metadata names (different naming conventions)
meta_names = meta['player'].tolist()
meta_map = {}
for _, row in squads.iterrows():
    p = row['player']
    result = process.extractOne(p, meta_names, scorer=fuzz.WRatio, score_cutoff=80)
    if result:
        meta_map[p] = result[0]

print(f"Matched {len(meta_map)}/{len(squads)} players to metadata")

squads['meta_match'] = squads['player'].map(meta_map)
meta_cols = ['player', 'batting_hand', 'bowling_style', 'bowling_type', 'role',
             'avg_batting_pos', 'total_runs', 'avg', 'sr', 'total_wickets', 'economy']
squads = squads.merge(meta[meta_cols].rename(columns={'player': 'meta_match'}),
                       on='meta_match', how='left', suffixes=('', '_meta'))

# ── Merge scouting profiles ──
scout_names = scouting['player'].tolist()
scout_map = {}
for _, row in squads.iterrows():
    p = row['player']
    result = process.extractOne(p, scout_names, scorer=fuzz.WRatio, score_cutoff=80)
    if result:
        scout_map[p] = result[0]

print(f"Matched {len(scout_map)}/{len(squads)} players to scouting profiles")

scout_cols = ['player', 'scouting_score', 'performance_rating', 'form_rating',
              'consistency_rating', 'impact_rating', 'pressure_rating',
              'playstyle', 'player_type', 'batting_role', 'bowling_category',
              'bowling_specialty', 'nationality', 'is_bowler', 'is_allrounder',
              'career_avg', 'career_sr', 'form_score', 'form_sr',
              'bowl_wickets', 'bowl_economy', 'bowl_avg', 'status']
scout_available = [c for c in scout_cols if c in scouting.columns]

squads['scout_match'] = squads['player'].map(scout_map)
squads = squads.merge(scouting[scout_available].rename(columns={'player': 'scout_match'}),
                       on='scout_match', how='left', suffixes=('', '_scout'))

# ── Fill defaults for unmatched players ──
squads['scouting_score'] = squads['scouting_score'].fillna(15.0)
squads['performance_rating'] = squads['performance_rating'].fillna(15.0)
squads['form_rating'] = squads['form_rating'].fillna(20.0)
squads['consistency_rating'] = squads['consistency_rating'].fillna(20.0)
squads['impact_rating'] = squads['impact_rating'].fillna(10.0)
squads['pressure_rating'] = squads['pressure_rating'].fillna(10.0)
squads['nationality'] = squads['nationality'].fillna(
    squads['is_overseas'].apply(lambda x: 'Overseas' if x else 'Indian'))
squads['seasons_played'] = squads.get('seasons_played', pd.Series([1]*len(squads))).fillna(1).astype(int)

# ── Verify ──
print(f"\n── Per-team summary ──")
for team in sorted(squads['current_franchise'].unique()):
    t = squads[squads['current_franchise'] == team]
    scouted = t['scouting_score'].gt(15).sum()
    ovs = t['is_overseas'].sum()
    print(f"  {team:35s}: {len(t):2d} players, {scouted:2d} scouted, {ovs:.0f} overseas")

In [ ]:
# ══════════════════════════════════════════════════════════════
# FIX: CORRECT BOWLING STYLES FOR KNOWN PLAYERS
# ══════════════════════════════════════════════════════════════

# Manual bowling style overrides for key players across all squads
# Format: 'Player Name': ('style_code', 'category')
# Style codes: RF=Right Fast, RFM=Right Fast-Med, LF=Left Fast, LFM=Left Fast-Med
#              OB=Off Break, LB=Leg Break, SLA=Slow Left Arm, SLAC=SLA Chinaman
BOWLING_OVERRIDES = {
    # ── Spinners ──
    'Rashid Khan': ('LB', 'Leg Spin'),
    'Yuzvendra Chahal': ('LB', 'Leg Spin'),
    'Varun Chakaravarthy': ('OB', 'Off Spin'),
    'Kuldeep Yadav': ('SLAC', 'Leg Spin'),       # Left-arm wrist spin = leg spin category
    'R Sai Kishore': ('SLA', 'Left-arm Spin'),
    'Noor Ahmad': ('SLAC', 'Leg Spin'),           # Left-arm wrist spin
    'Washington Sundar': ('OB', 'Off Spin'),
    'Sunil Narine': ('OB', 'Off Spin'),
    'Rachin Ravindra': ('SLA', 'Left-arm Spin'),
    'Axar Patel': ('SLA', 'Left-arm Spin'),
    'Manav Suthar': ('SLA', 'Left-arm Spin'),
    'Shreyas Gopal': ('LB', 'Leg Spin'),
    'Wanindu Hasaranga': ('LB', 'Leg Spin'),
    'Mayank Markande': ('LB', 'Leg Spin'),
    'Akeal Hosein': ('SLA', 'Left-arm Spin'),
    'Shahbaz Ahmed': ('SLA', 'Left-arm Spin'),
    'Jayant Yadav': ('OB', 'Off Spin'),
    'Anukul Roy': ('SLA', 'Left-arm Spin'),
    'Swapnil Singh': ('SLA', 'Left-arm Spin'),
    'Harpreet Brar': ('SLA', 'Left-arm Spin'),
    'Zeeshan Ansari': ('SLA', 'Left-arm Spin'),
    'Prashant Solanki': ('LB', 'Leg Spin'),
    'Kamindu Mendis': ('SLA', 'Left-arm Spin'),
    'Maheesh Theekshana': ('OB', 'Off Spin'),
    'Liam Livingstone': ('LB', 'Leg Spin'),
    'Aman Khan': ('LB', 'Leg Spin'),
    'Rahul Chahar': ('LB', 'Leg Spin'),
    'Suyash Sharma': ('LB', 'Leg Spin'),
    'AJ Hosein': ('SLA', 'Left-arm Spin'),
    'Vicky Ostwal': ('SLA', 'Left-arm Spin'),
    
    # ── Left-arm pace ──
    'Arshdeep Singh': ('LFM', 'Left-arm Fast'),
    'T Natarajan': ('LFM', 'Left-arm Fast'),
    'Khaleel Ahmed': ('LFM', 'Left-arm Fast'),
    'Mukesh Choudhary': ('LFM', 'Left-arm Fast'),
    'Marco Jansen': ('LFM', 'Left-arm Fast'),
    'Trent Boult': ('LFM', 'Left-arm Fast'),
    'Mitchell Starc': ('LF', 'Left-arm Fast'),
    'Sam Curran': ('LFM', 'Left-arm Fast'),
    'Mohsin Khan': ('LFM', 'Left-arm Fast'),
    'Yash Dayal': ('LFM', 'Left-arm Fast'),
    'Kwena Maphaka': ('LF', 'Left-arm Fast'),
    'Akash Singh': ('LFM', 'Left-arm Fast'),
    'Jaydev Unadkat': ('LFM', 'Left-arm Fast'),
    'Gurjapneet Singh': ('LFM', 'Left-arm Fast'),
    
    # ── Right-arm pace ──
    'Jasprit Bumrah': ('RF', 'Fast'),
    'Kagiso Rabada': ('RF', 'Fast'),
    'Mitchell Santner': ('SLA', 'Left-arm Spin'),  # he's actually a spinner
    'Pat Cummins': ('RF', 'Fast'),
    'Josh Hazlewood': ('RFM', 'Fast'),
    'Lockie Ferguson': ('RF', 'Fast'),
    'Matheesha Pathirana': ('RF', 'Fast'),
    'Jofra Archer': ('RF', 'Fast'),
    'Mohammed Shami': ('RFM', 'Fast'),
    'Prasidh Krishna': ('RF', 'Fast'),
    'Deepak Chahar': ('RFM', 'Fast'),
    'Harshal Patel': ('RFM', 'Fast'),
    'Bhuvneshwar Kumar': ('RFM', 'Fast'),
    'Umran Malik': ('RF', 'Fast'),
    'Avesh Khan': ('RFM', 'Fast'),
    'Mayank Yadav': ('RF', 'Fast'),
    'Brydon Carse': ('RFM', 'Fast'),
    'Nathan Ellis': ('RFM', 'Fast'),
    'Harshit Rana': ('RFM', 'Fast'),
    'Rasikh Salam': ('RFM', 'Fast'),
    'Sandeep Sharma': ('RFM', 'Fast'),
    'Mohammed Siraj': ('RFM', 'Fast'),
    'Mukesh Kumar': ('RFM', 'Fast'),
    'Anshul Kamboj': ('RFM', 'Fast'),
    'Xavier Bartlett': ('RFM', 'Fast'),
    'Matt Henry': ('RFM', 'Fast'),
    'Kyle Jamieson': ('RFM', 'Fast'),
    'Lungi Ngidi': ('RF', 'Fast'),
    'Nandre Burger': ('RFM', 'Fast'),
    'Tushar Deshpande': ('RFM', 'Fast'),
    'Vyshak Vijaykumar': ('RFM', 'Fast'),
    'Blessing Muzarabani': ('RF', 'Fast'),
    'Ishant Sharma': ('RFM', 'Fast'),
    'Shardul Thakur': ('RFM', 'Fast'),
    'Nuwan Thushara': ('RF', 'Fast'),
    
    # ── All-rounders (bowling style) ──
    'Hardik Pandya': ('RFM', 'Fast'),
    'Shivam Dube': ('RFM', 'Fast'),
    'Cameron Green': ('RFM', 'Fast'),
    'Jamie Overton': ('RFM', 'Fast'),
    'Romario Shepherd': ('RFM', 'Fast'),
    'Nitish Kumar Reddy': ('RFM', 'Fast'),
    'Venkatesh Iyer': ('RFM', 'Fast'),
    'Marcus Stoinis': ('RFM', 'Fast'),
    'Jason Holder': ('RFM', 'Fast'),
    'Krunal Pandya': ('SLA', 'Left-arm Spin'),
    'Dewald Brevis': ('LB', 'Leg Spin'),
    'Abhishek Sharma': ('SLA', 'Left-arm Spin'),
    'Prashant Veer': ('OB', 'Off Spin'),
    'Matthew Short': ('OB', 'Off Spin'),
    'Azmatullah Omarzai': ('RFM', 'Fast'),
}

# Apply overrides
for player_name, (style, category) in BOWLING_OVERRIDES.items():
    mask = squads['player'] == player_name
    if mask.any():
        squads.loc[mask, 'bowling_style'] = style
        squads.loc[mask, 'bowling_category'] = category

# Verify fix
print("── Bowling style check per team ──")
for team in sorted(squads['current_franchise'].unique()):
    bowlers = squads[(squads['current_franchise'] == team) & 
                     (squads['clean_role'].isin(['Bowler', 'All-Rounder']))]
    styles = bowlers['bowling_style'].value_counts().to_dict()
    print(f"  {team:35s}: {styles}")

In [ ]:
# ── Role-normalized performance ──
# Bowlers and batters are scored on different scales in scouting_profiles.
# Normalize within each role group to 0-100 so a top bowler = top batter.
def role_normalize(df, col, role_col='clean_role'):
    """Normalize a column within each role group using percentile ranking."""
    result = df[col].copy()
    for role in df[role_col].unique():
        mask = df[role_col] == role
        vals = df.loc[mask, col]
        if vals.notna().sum() > 1:
            # Percentile rank within role → 0-100
            pct = vals.rank(pct=True) * 100
            result.loc[mask] = pct
    return result

squads['norm_performance'] = role_normalize(squads, 'performance_rating')
squads['norm_form'] = role_normalize(squads, 'form_rating')

# Verify the fix works: top bowlers should now score ~90-100
print("── Normalized performance: top bowlers ──")
bowlers = squads[squads['clean_role'] == 'Bowler'].nlargest(5, 'norm_performance')
print(bowlers[['player', 'current_franchise', 'scouting_score', 'performance_rating', 
               'norm_performance']].to_string(index=False))

print("\n── Normalized performance: top batters ──")
batters = squads[squads['clean_role'] == 'Batter'].nlargest(5, 'norm_performance')
print(batters[['player', 'current_franchise', 'scouting_score', 'performance_rating', 
               'norm_performance']].to_string(index=False))

# ── Fix nationality misclassifications ──
OVERSEAS_OVERRIDE = [
    'RD Rickelton', 'D Brevis', 'BKG Mendis', 'PHKD Mendis', 
    'AF Milne', 'NT Ellis', 'M Pathirana', 'Mujeeb Ur Rahman',
    'AJ Hosein', 'FA Allen', 'JG Bethell', 'PD Salt', 'PWA Mulder',
    'PWH de Silva', 'RJW Topley', 'KT Maphaka', 'MJ Henry',
]
for p in OVERSEAS_OVERRIDE:
    squads.loc[squads['player'] == p, 'nationality'] = 'Overseas'
    squads.loc[squads['player'] == p, 'is_overseas'] = True

print(f"\n── Overseas count per team (after fix) ──")
for team in sorted(squads['current_franchise'].unique()):
    t = squads[squads['current_franchise'] == team]
    ovs = t['is_overseas'].sum()
    print(f"  {team:35s}: {ovs:.0f} overseas")

# ── Improved price efficiency ──
# Instead of raw ratio, use a curve that doesn't crush expensive stars.
# The idea: a 18 Cr player with 90th percentile performance is GOOD value.
# A 18 Cr player with 30th percentile performance is BAD value.
def compute_price_efficiency_v2(row):
    """
    Compare normalized performance percentile vs price percentile.
    If your performance rank > price rank, you're good value.
    """
    perf_pct = row.get('norm_performance', 50)
    price = row.get('price_cr', 0.30)
    
    # Price percentile among all active players
    # We'll compute this outside and pass in
    price_pct = row.get('price_percentile', 50)
    
    # Efficiency = how much your performance outpaces your price
    # perf_pct 90, price_pct 80 → 60 (good)
    # perf_pct 30, price_pct 80 → 25 (overpaid)
    # perf_pct 90, price_pct 20 → 85 (steal)
    diff = perf_pct - price_pct
    efficiency = np.clip(50 + diff * 0.5, 0, 100)
    return round(efficiency, 1)

# Compute price percentile
squads['price_percentile'] = squads['price_cr'].rank(pct=True) * 100
squads['price_efficiency_v2'] = squads.apply(compute_price_efficiency_v2, axis=1)

# ── Replaceability (same as before) ──
def compute_replaceability(player_row, scouting_df):
    role = player_row.get('clean_role', 'Batter')
    is_overseas = player_row.get('is_overseas', False)
    
    pool = scouting_df[scouting_df['status'].isin(['Active', 'Unsold', 'International'])] if 'status' in scouting_df.columns else scouting_df
    
    if role == 'Bowler':
        similar_pool = pool[pool.get('is_bowler', pd.Series([False]*len(pool))) == True]
    elif role == 'All-Rounder':
        similar_pool = pool[pool.get('is_allrounder', pd.Series([False]*len(pool))) == True]
    else:
        similar_pool = pool[(pool.get('is_bowler', pd.Series([False]*len(pool))) == False) & 
                           (pool.get('is_allrounder', pd.Series([False]*len(pool))) == False)]
    
    if 'nationality' in similar_pool.columns:
        if is_overseas:
            similar_pool = similar_pool[similar_pool['nationality'] == 'Overseas']
        else:
            similar_pool = similar_pool[similar_pool['nationality'] == 'Indian']
    
    count = len(similar_pool)
    return min(count / 100 * 100, 100)

# ── Squad importance (same as before) ──
def compute_squad_importance(player_row, team_df):
    role = player_row['clean_role']
    same_role_count = len(team_df[team_df['clean_role'] == role])
    importance_map = {1: 100, 2: 75, 3: 50, 4: 30, 5: 15}
    return importance_map.get(same_role_count, 10)

# ── Compute retention value (v2) ──
retention_rows = []

for team in sorted(squads['current_franchise'].unique()):
    team_df = squads[squads['current_franchise'] == team]
    
    for idx, row in team_df.iterrows():
        perf = row.get('norm_performance', 50)
        form = row.get('norm_form', 50)
        price_eff = row.get('price_efficiency_v2', 50)
        replaceability = compute_replaceability(row, scouting)
        irreplaceability = 100 - replaceability
        squad_imp = compute_squad_importance(row, team_df)
        
        retention_value = (
            perf * 0.30 +
            form * 0.20 +
            price_eff * 0.15 +
            irreplaceability * 0.20 +
            squad_imp * 0.15
        )
        
        retention_rows.append({
            'player': row['player'],
            'team': row['current_franchise'],
            'price_cr': row['price_cr'],
            'clean_role': row['clean_role'],
            'is_overseas': row['is_overseas'],
            'scouting_score': row['scouting_score'],
            'norm_performance': round(perf, 1),
            'norm_form': round(form, 1),
            'price_efficiency': round(price_eff, 1),
            'irreplaceability': round(irreplaceability, 1),
            'squad_importance': round(squad_imp, 1),
            'retention_value': round(retention_value, 1),
        })

ret_df = pd.DataFrame(retention_rows)

# ── Classify ──
def classify_retention(team_df):
    team_df = team_df.sort_values('retention_value', ascending=False).copy()
    team_df['rank'] = range(1, len(team_df) + 1)
    
    labels = []
    for _, row in team_df.iterrows():
        rv = row['retention_value']
        rank = row['rank']
        price = row['price_cr']
        perf = row['norm_performance']
        
        if rank <= 5:
            labels.append('RETAIN')
        elif rank <= 10 and rv > 45:
            labels.append('RTM')
        elif rv > 35 and price > 5.0 and perf < 60:
            # Decent retention value but high price + mediocre performance = overpaid
            labels.append('BUY BACK')
        else:
            labels.append('RELEASE')
    
    team_df['recommendation'] = labels
    return team_df

ret_classified = ret_df.groupby('team', group_keys=False).apply(classify_retention)

# ── Output ──
print("\n── Recommendation distribution ──")
print(ret_classified['recommendation'].value_counts())

print("\n── Mumbai Indians ──")
mi = ret_classified[ret_classified['team'] == 'Mumbai Indians'][
    ['player', 'price_cr', 'clean_role', 'is_overseas', 'norm_performance', 
     'retention_value', 'recommendation']
].sort_values('retention_value', ascending=False)
print(mi.to_string(index=False))

print("\n── Chennai Super Kings ──")
csk = ret_classified[ret_classified['team'] == 'Chennai Super Kings'][
    ['player', 'price_cr', 'clean_role', 'is_overseas', 'norm_performance', 
     'retention_value', 'recommendation']
].sort_values('retention_value', ascending=False)
print(csk.to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL — DEEP SQUAD GAPS ANALYSIS (FINAL)
# ══════════════════════════════════════════════════════════════

# ── Merge phase-wise stats from scouting_profiles into squads ──
phase_cols = ['player', 'avg_runs_Death', 'avg_runs_Middle', 'avg_runs_Powerplay',
              'avg_sr_Death', 'avg_sr_Middle', 'avg_sr_Powerplay',
              'bowl_economy_po', 'bowl_economy_mi', 'bowl_economy_de',
              'bowl_wickets_po', 'bowl_wickets_mi', 'bowl_wickets_de',
              'bowl_dot_pct', 'boundary_rate', 'high_pressure_avg',
              'pace_sr', 'spin_sr', 'consistency']

phase_available = [c for c in phase_cols if c in scouting.columns]
phase_data = scouting[phase_available].drop_duplicates(subset='player')

# Only merge if not already merged
if 'avg_sr_Death' not in squads.columns:
    squads = squads.merge(phase_data, left_on='scout_match', right_on='player',
                          how='left', suffixes=('', '_phase'))
    if 'player_phase' in squads.columns:
        squads.drop(columns=['player_phase'], inplace=True)
    print(f"Merged phase data — squads now has {squads.shape[1]} columns")
else:
    print("Phase data already merged")

# ── Load master for team-level historical analysis ──
master = pd.read_csv('../data/master_ipl.csv', low_memory=False)
print(f"Master data: {len(master)} deliveries")

# ── Compute league-wide benchmarks (last 3 seasons) ──
recent = master[master['season'].isin([2023, 2024, 2025])].copy()

def get_phase(over):
    if over < 6: return 'powerplay'
    elif over < 16: return 'middle'
    else: return 'death'

recent['phase'] = recent['over'].apply(get_phase)

league_benchmarks = {}
for phase in ['powerplay', 'middle', 'death']:
    ph = recent[recent['phase'] == phase]
    total_runs = ph['total_runs'].sum()
    total_balls = len(ph)
    league_sr = (total_runs / total_balls) * 100 if total_balls > 0 else 0
    league_eco = (total_runs / total_balls) * 6 if total_balls > 0 else 0
    total_wickets = ph['is_wicket'].sum()
    total_overs = total_balls / 6
    league_benchmarks[phase] = {
        'sr': round(league_sr, 1),
        'economy': round(league_eco, 2),
        'wickets_per_over': round(total_wickets / total_overs, 3) if total_overs > 0 else 0
    }

print(f"\n── League benchmarks (2023-2025) ──")
for phase, stats in league_benchmarks.items():
    print(f"  {phase:12s}: SR={stats['sr']}, Eco={stats['economy']}, W/Over={stats['wickets_per_over']}")

# ── Team historical performance ──
def compute_team_history(team_name, master_df):
    team_batting = master_df[(master_df['batting_team'] == team_name) &
                             (master_df['season'].isin([2024, 2025]))].copy()
    team_bowling = master_df[(master_df['bowling_team'] == team_name) &
                             (master_df['season'].isin([2024, 2025]))].copy()
    if len(team_batting) == 0:
        return None
    team_batting['phase'] = team_batting['over'].apply(get_phase)
    team_bowling['phase'] = team_bowling['over'].apply(get_phase)
    history = {}
    for phase in ['powerplay', 'middle', 'death']:
        bp = team_batting[team_batting['phase'] == phase]
        bowlp = team_bowling[team_bowling['phase'] == phase]
        bat_sr = (bp['total_runs'].sum() / len(bp) * 100) if len(bp) > 0 else 0
        bowl_eco = (bowlp['total_runs'].sum() / len(bowlp) * 6) if len(bowlp) > 0 else 0
        history[phase] = {'bat_sr': round(bat_sr, 1), 'bowl_eco': round(bowl_eco, 2)}
    match_col = 'matchId' if 'matchId' in master_df.columns else 'match_id'
    if match_col in master_df.columns:
        matches = master_df[((master_df['batting_team'] == team_name) |
                              (master_df['bowling_team'] == team_name)) &
                             (master_df['season'].isin([2024, 2025]))]
        history['matches_in_data'] = len(matches[match_col].unique())
    else:
        history['matches_in_data'] = 0
    return history


# ── Deep analysis function ──
def deep_squad_analysis_final(team_name, squad_df, ret_df, scouting_df, master_df, benchmarks):
    team = squad_df[squad_df['current_franchise'] == team_name].copy()
    team_ret = ret_df[ret_df['team'] == team_name].copy()

    gaps = []
    strengths = []

    # ═══════════════════════════════════
    # 1. BATTING QUALITY
    # ═══════════════════════════════════

    openers = team[team['batting_role'] == 'Opener']
    if len(openers) > 0:
        avg_opener_avg = openers['career_avg'].mean()
        avg_opener_sr = openers['career_sr'].mean()
        if pd.notna(avg_opener_avg) and avg_opener_avg < 25:
            gaps.append(f"Weak opening avg ({avg_opener_avg:.1f}) — need 25+")
        elif pd.notna(avg_opener_avg) and avg_opener_avg > 32:
            strengths.append(f"Strong openers (avg {avg_opener_avg:.1f})")
        if pd.notna(avg_opener_sr) and avg_opener_sr < 125:
            gaps.append(f"Openers lack aggression (SR {avg_opener_sr:.1f})")
    else:
        gaps.append("No designated openers")

    # Explosiveness
    all_bats = team[team['clean_role'].isin(['Batter', 'WK-Batter'])]
    explosive = all_bats[all_bats['career_sr'] > 140]
    if len(all_bats) > 0:
        if len(explosive) < 2:
            gaps.append(f"Lack of explosiveness — only {len(explosive)}/{len(all_bats)} batters with SR > 140")
        elif len(explosive) >= len(all_bats) * 0.5:
            strengths.append(f"Explosive lineup ({len(explosive)}/{len(all_bats)} SR > 140)")

    # Death batting
    death_bats = team[team['avg_sr_Death'].notna() & (team['avg_sr_Death'] > 50)]
    if len(death_bats) >= 3:
        team_death_sr = death_bats['avg_sr_Death'].mean()
        if team_death_sr < 130:
            gaps.append(f"Weak death batting (avg SR {team_death_sr:.0f})")
        elif team_death_sr > 155:
            strengths.append(f"Strong death batting (avg SR {team_death_sr:.0f})")

    # Finisher
    finishers = team[team['batting_role'] == 'Finisher']
    ar_finishers = team[(team['clean_role'] == 'All-Rounder') &
                        (team['avg_sr_Death'].fillna(0) > 145)]
    if len(finishers) == 0 and len(ar_finishers) == 0:
        gaps.append("No designated finisher for death overs")

    # Left-right balance
    top_order = team[team['batting_role'].isin(['Opener', 'Top Order', 'Middle Order'])]
    if len(top_order) > 0:
        left_bats = len(top_order[top_order['batting_hand'] == 'L'])
        if left_bats == 0:
            gaps.append("All right-handed top order — easy for bowlers to set one plan")
        elif left_bats >= 2:
            strengths.append(f"Good L-R balance ({left_bats} lefties in top order)")

    # ═══════════════════════════════════
    # 2. BOWLING QUALITY
    # ═══════════════════════════════════

    bowling_squad = team[team['clean_role'].isin(['Bowler', 'All-Rounder'])]
    styles = bowling_squad['bowling_style'].dropna().str.strip().tolist()

    style_categories = {
        'Right-arm fast': [s for s in styles if s.upper() in ['RFM', 'RF']],
        'Left-arm fast': [s for s in styles if s.upper() in ['LFM', 'LF', 'LMF']],
        'Off spin': [s for s in styles if s.upper() in ['OB', 'ROB']],
        'Leg spin': [s for s in styles if s.upper() in ['LB', 'LBG', 'LWS', 'SLAC']],
        'Left-arm spin': [s for s in styles if s.upper() in ['SLA', 'SLAO']],
    }
    bowling_variety = {k: len(v) for k, v in style_categories.items()}

    # Death bowling
    death_bowlers = bowling_squad[bowling_squad['bowl_economy_de'].notna()]
    if len(death_bowlers) > 0:
        good_death = death_bowlers[death_bowlers['bowl_economy_de'] < 9.5]
        if len(good_death) < 2:
            gaps.append(f"Death bowling weakness — only {len(good_death)} bowlers with death eco < 9.5")
        else:
            strengths.append(f"Reliable death bowling ({len(good_death)} bowlers with eco < 9.5)")

    # Powerplay wickets
    pp_bowlers = bowling_squad[bowling_squad['bowl_wickets_po'].notna()]
    if len(pp_bowlers) > 0:
        total_pp_w = pp_bowlers['bowl_wickets_po'].sum()
        if total_pp_w < 20:
            gaps.append(f"Low PP wicket-taking ({total_pp_w:.0f} total PP wickets across squad)")

    # Bowling variety gaps
    if bowling_variety.get('Left-arm fast', 0) == 0 and bowling_variety.get('Left-arm spin', 0) == 0:
        gaps.append("No left-arm bowling option")
    if bowling_variety.get('Leg spin', 0) == 0:
        gaps.append("No leg spinner — missing wrist spin")
    if bowling_variety.get('Off spin', 0) == 0 and bowling_variety.get('Left-arm spin', 0) == 0:
        gaps.append("No finger spinner — vulnerable on turning tracks")

    # Bowling variety strength (raised threshold)
    total_styles = sum(1 for v in bowling_variety.values() if v > 0)
    if total_styles == 5:
        strengths.append("Complete bowling variety — all 5 types covered")
    elif total_styles == 4:
        missing = [k for k, v in bowling_variety.items() if v == 0]
        strengths.append(f"Good bowling variety (4/5 types, missing: {missing[0]})")

    # ═══════════════════════════════════
    # 3. ADDITIONAL STRENGTH CHECKS
    # ═══════════════════════════════════

    # Star power
    elite = team[team['scouting_score'] > 60]
    if len(elite) >= 5:
        strengths.append(f"Deep star power — {len(elite)} players with scouting score above 60")
    elif len(elite) >= 3:
        top_names = elite.nlargest(3, 'scouting_score')['player'].tolist()
        strengths.append(f"Solid core — {len(elite)} elite players (inc. {', '.join(top_names[:2])})")

    # Indian core
    indian_quality = team[(~team['is_overseas']) & (team['scouting_score'] > 50)]
    if len(indian_quality) >= 5:
        strengths.append(f"Strong Indian core — {len(indian_quality)} quality Indian players")

    # All-rounder depth
    ars = team[team['clean_role'] == 'All-Rounder']
    quality_ars = ars[ars['scouting_score'] > 40]
    if len(quality_ars) >= 3:
        strengths.append(f"All-rounder depth — {len(quality_ars)} quality ARs give flexibility")

    # Pace battery
    pace_bowlers = bowling_squad[bowling_squad['bowling_style'].fillna('').str.upper().isin(
        ['RF', 'RFM', 'LF', 'LFM', 'LMF'])]
    quality_pace = pace_bowlers[pace_bowlers['scouting_score'] > 35]
    if len(quality_pace) >= 4:
        strengths.append(f"Deep pace battery — {len(quality_pace)} quality pacers")

    # Spin depth
    spin_bowlers = bowling_squad[bowling_squad['bowling_style'].fillna('').str.upper().isin(
        ['OB', 'LB', 'LBG', 'SLA', 'SLAC', 'LWS', 'ROB'])]
    if len(spin_bowlers) >= 3:
        strengths.append(f"Spin-rich squad — {len(spin_bowlers)} spinners for turning tracks")

    # Young talent
    if 'seasons_played' in team.columns:
        young = team[team['seasons_played'].fillna(0) <= 2]
        scouted_young = young[young['scouting_score'] > 40]
        if len(scouted_young) >= 3:
            strengths.append(f"Strong youth pipeline — {len(scouted_young)} talented young players")

    # Bench strength
    bench = team[team['scouting_score'] > 30]
    if len(bench) >= 18:
        strengths.append(f"Excellent bench strength — {len(bench)}/{len(team)} players with scouting 30+")
    elif len(bench) >= 14:
        strengths.append(f"Good squad depth — {len(bench)}/{len(team)} players with scouting 30+")

    # Value picks
    bargains = team[(team['scouting_score'] > 50) & (team['price_cr'] < 2.0)]
    if len(bargains) >= 3:
        names = bargains.nlargest(2, 'scouting_score')['player'].tolist()
        strengths.append(f"Value picks — {len(bargains)} high-quality players under ₹2 Cr (inc. {', '.join(names)})")

    # ═══════════════════════════════════
    # 4. DEPENDENCY ANALYSIS
    # ═══════════════════════════════════

    n_overseas = int(team['is_overseas'].sum())
    overseas_bats = len(team[(team['is_overseas']) & (team['clean_role'].isin(['Batter', 'WK-Batter']))])
    overseas_bowlers = len(team[(team['is_overseas']) & (team['clean_role'] == 'Bowler')])
    indian_bowlers = len(team[(~team['is_overseas']) & (team['clean_role'] == 'Bowler')])

    if overseas_bowlers == 0 and n_overseas >= 4:
        gaps.append("No overseas bowler in squad")
    if overseas_bats >= 4:
        gaps.append(f"{overseas_bats} overseas batters — limits XI selection flexibility")
    if indian_bowlers < 3:
        gaps.append(f"Only {indian_bowlers} Indian bowlers — need 3+ for XI flexibility")

    # Quality drop-off
    scored_players = team[team['career_avg'].notna() & (team['career_avg'] > 0)]
    if len(scored_players) >= 3:
        top = scored_players.nlargest(1, 'career_avg')
        second = scored_players.nlargest(2, 'career_avg').iloc[-1]
        if top.iloc[0]['career_avg'] > second['career_avg'] * 1.6:
            gaps.append(f"Batting quality drop-off after {top.iloc[0]['player']} "
                        f"(avg {top.iloc[0]['career_avg']:.0f} vs next best {second['career_avg']:.0f})")

    # Wicket dependency
    wk_players = bowling_squad[bowling_squad['bowl_wickets'].notna() & (bowling_squad['bowl_wickets'] > 0)]
    if len(wk_players) >= 2:
        top_w = wk_players.nlargest(1, 'bowl_wickets')
        total_w = wk_players['bowl_wickets'].sum()
        if total_w > 0:
            pct = top_w.iloc[0]['bowl_wickets'] / total_w * 100
            if pct > 40:
                gaps.append(f"Wicket dependency on {top_w.iloc[0]['player']} ({pct:.0f}% of squad wickets)")

    # ═══════════════════════════════════
    # 5. COMPOSITION
    # ═══════════════════════════════════

    roles = team['clean_role'].value_counts().to_dict()
    if roles.get('WK-Batter', 0) == 0:
        gaps.append("No specialist wicket-keeper!")
    if roles.get('All-Rounder', 0) < 2:
        gaps.append(f"Only {roles.get('All-Rounder', 0)} all-rounder(s) — need 2+")
    if n_overseas > 8:
        gaps.append(f"{n_overseas} overseas — exceeds squad limit of 8")
    if len(team) < 18:
        gaps.append(f"Small squad ({len(team)}) — injury risk")

    # ═══════════════════════════════════
    # 6. RETENTION + PURSE
    # ═══════════════════════════════════

    retained = team_ret[team_ret['recommendation'] == 'RETAIN']
    rtm = team_ret[team_ret['recommendation'] == 'RTM']
    released = team_ret[team_ret['recommendation'].isin(['RELEASE', 'BUY BACK'])]

    retain_cost = team[team['player'].isin(retained['player'])]['price_cr'].sum()
    rtm_cost = team[team['player'].isin(rtm['player'])]['price_cr'].sum()
    total_spend = team['price_cr'].sum()

    kept = team[team['player'].isin(retained['player'].tolist() + rtm['player'].tolist())]
    kept_roles = kept['clean_role'].value_counts().to_dict()

    priority_buys = []
    if kept_roles.get('Bowler', 0) < 3:
        priority_buys.append({'role': 'Bowler', 'priority': 'HIGH',
                              'current': kept_roles.get('Bowler', 0), 'need': 3})
    if kept_roles.get('Batter', 0) + kept_roles.get('WK-Batter', 0) < 3:
        bats = kept_roles.get('Batter', 0) + kept_roles.get('WK-Batter', 0)
        priority_buys.append({'role': 'Batter', 'priority': 'HIGH', 'current': bats, 'need': 3})
    if kept_roles.get('All-Rounder', 0) < 1:
        priority_buys.append({'role': 'All-Rounder', 'priority': 'MEDIUM',
                              'current': kept_roles.get('All-Rounder', 0), 'need': 1})
    if kept_roles.get('WK-Batter', 0) < 1:
        priority_buys.append({'role': 'WK-Batter', 'priority': 'HIGH', 'current': 0, 'need': 1})

    # ═══════════════════════════════════
    # 7. HISTORICAL TRENDS
    # ═══════════════════════════════════

    history = compute_team_history(team_name, master_df)
    historical_notes = []
    if history and history.get('matches_in_data', 0) > 0:
        for phase in ['powerplay', 'middle', 'death']:
            if phase in history:
                bat_sr = history[phase]['bat_sr']
                bowl_eco = history[phase]['bowl_eco']
                if bat_sr < benchmarks[phase]['sr'] - 8:
                    historical_notes.append(f"Below-avg {phase} batting (SR {bat_sr} vs {benchmarks[phase]['sr']})")
                if bowl_eco > benchmarks[phase]['economy'] + 0.8:
                    historical_notes.append(f"Leaky {phase} bowling (eco {bowl_eco} vs {benchmarks[phase]['economy']})")

    # ── History-dependent strengths (must come after history is computed) ──
    if history and history.get('matches_in_data', 0) > 0:
        if 'powerplay' in history:
            if history['powerplay']['bat_sr'] > benchmarks['powerplay']['sr'] + 5:
                strengths.append(f"Aggressive PP batting (SR {history['powerplay']['bat_sr']} vs {benchmarks['powerplay']['sr']} league)")
        if 'middle' in history:
            if history['middle']['bowl_eco'] < benchmarks['middle']['economy'] - 0.5:
                strengths.append(f"Tight middle-overs bowling (eco {history['middle']['bowl_eco']} vs {benchmarks['middle']['economy']} league)")
        if 'death' in history:
            if history['death']['bowl_eco'] < benchmarks['death']['economy'] - 0.5:
                strengths.append(f"Strong death bowling (eco {history['death']['bowl_eco']} vs {benchmarks['death']['economy']} league)")

    return {
        'team': team_name,
        'squad_size': len(team),
        'composition': dict(roles),
        'overseas_count': n_overseas,
        'bowling_variety': bowling_variety,
        'total_spend': round(total_spend, 2),
        'retain_cost': round(retain_cost, 2),
        'rtm_cost': round(rtm_cost, 2),
        'released_savings': round(total_spend - retain_cost - rtm_cost, 2),
        'post_retention_roles': dict(kept_roles),
        'strengths': strengths,
        'gaps': gaps,
        'priority_buys': priority_buys,
        'historical_notes': historical_notes,
        'retain_count': len(retained),
        'rtm_count': len(rtm),
        'release_count': len(released),
    }


# ── Run for all teams ──
squad_analysis = {}
for team in sorted(squads['current_franchise'].unique()):
    squad_analysis[team] = deep_squad_analysis_final(team, squads, ret_classified, scouting, master, league_benchmarks)

for team, a in squad_analysis.items():
    print(f"\n{'═' * 70}")
    print(f"  {team}")
    print(f"{'═' * 70}")
    print(f"  Squad: {a['squad_size']} | {a['composition']} | {a['overseas_count']} ovs")
    print(f"  Bowling: {a['bowling_variety']}")
    print(f"  Spend: {a['total_spend']} → Retain: {a['retain_cost']} | RTM: {a['rtm_cost']} | "
          f"Freed: {a['released_savings']} Cr")
    print(f"  After retention: {a['post_retention_roles']}")
    for s in a['strengths']:
        print(f"  ✓ {s}")
    for g in a['gaps']:
        print(f"  ✗ {g}")
    for h in a['historical_notes']:
        print(f"  📊 {h}")
    if a['priority_buys']:
        print(f"  → BUY: {', '.join(b['role'] + ' (' + b['priority'] + ')' for b in a['priority_buys'])}")


In [ ]:
# ══════════════════════════════════════════════════════════════
# SAVE CELL — AUCTION INTELLIGENCE DATA
# ══════════════════════════════════════════════════════════════
import json

# ── 1. Save team squads with retention recommendations ──
ret_save = ret_classified[['player', 'team', 'retention_value', 'recommendation', 'rank',
                            'norm_performance', 'norm_form', 'price_efficiency',
                            'irreplaceability', 'squad_importance']].copy()

# Only keep columns that actually exist in squads
all_cols = list(squads.columns)
save_cols = [c for c in [
    'player', 'current_franchise', 'price_cr', 'clean_role', 'is_overseas', 
    'nationality', 'scouting_score', 'how_acquired', 'seasons_played',
    'career_avg', 'career_sr', 'form_score', 'form_sr',
    'bowl_wickets', 'bowl_economy', 'bowl_avg',
    'batting_hand', 'bowling_style', 'bowling_type',
    'batting_role', 'bowling_category', 'bowling_specialty',
    'role', 'playstyle', 'player_type',
    'performance_rating', 'form_rating', 'consistency_rating',
    'impact_rating', 'pressure_rating'
] if c in all_cols]

squad_save = squads[save_cols].copy()

# Merge retention recommendations
squad_final = squad_save.merge(ret_save, left_on=['player', 'current_franchise'], 
                                right_on=['player', 'team'], how='left')
squad_final.drop(columns=['team'], inplace=True, errors='ignore')

squad_final.to_csv('../data/team_squads.csv', index=False)
print(f"Saved team_squads.csv: {squad_final.shape}")

# ── 2. Save squad analysis as JSON ──
def clean_for_json(obj):
    if isinstance(obj, dict):
        return {k: clean_for_json(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [clean_for_json(i) for i in obj]
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

squad_analysis_clean = clean_for_json(squad_analysis)

with open('../data/squad_analysis.json', 'w') as f:
    json.dump(squad_analysis_clean, f, indent=2)
print(f"Saved squad_analysis.json: {len(squad_analysis_clean)} teams")

# ── 3. Quick verify ──
mi = squad_final[squad_final['current_franchise'] == 'Mumbai Indians'][
    ['player', 'price_cr', 'clean_role', 'is_overseas', 'retention_value', 'recommendation']
].sort_values('retention_value', ascending=False).head(10)
print(f"\n── MI top 10 ──")
print(mi.to_string(index=False))